# Data Science Competition FIK FAIR 2026 — Official Final Pipeline
## Regresi Tingkat Biaya Hidup Wilayah: Anonymous Competition Score (ACS)
### Champion Architecture: TabPFN v2 Dual-Representation Multi-Seed Ensemble (`T3_TA12_50`)

---

###  Ringkasan Eksekutif & Metodologi Inti

Notebook ini mengimplementasikan pipeline inferensi deterministik resmi untuk menghasilkan berkas submission final kompetisi **DSC FIK FAIR 2026**. Berdasarkan serangkaian audit empiris lintas model (dari baseline linear, ExtraTrees, Ridge residual, hingga representasi manifold non-linear), pipeline ini mengadopsi arsitektur model fondasi tabular **TabPFN v2** (*Prior-data Fitted Networks*) dengan formulasi fusi spasial **`T3_TA12_50`** yang membukukan rekor skor **Public Leaderboard 8.09**.

#### 1. Karakteristik Masalah & Batasan Regulasi
* **Dataset Terbatas ($N=382$ Latih, $N=96$ Uji)**: Rasio observasi terhadap fitur ($382:24$) sangat rentan terhadap *overfitting* dan *variance explosion* bila dimodelkan dengan model pohon berkapasitas besar (*deep tree ensembling*) atau regresi diferensial berderajat tinggi.
* **Fitur Anonim**: 24 prediktor numerik kontinu (`feature_01` s.d. `feature_24`) merepresentasikan parameter iklim dan lingkungan regional tanpa metadata geospasial eksplisit.
* **Target ACS**: Nilai metrik kontinu tanpa unit (rentang train: $[32.59, 100.44]$, $\mu=51.20, \sigma=12.91$) yang berbanding lurus dengan estimasi biaya hidup.
* **Metrik Evaluasi**: Evaluasi resmi dihitung menggunakan **Root Mean Squared Error (RMSE)** dengan penalti kuadratik terhadap galat prediksi, serta Mean Absolute Error (MAE) sebagai *tie-breaker*.

#### 2. Pilar Desain Arsitektur `T3_TA12_50`
Model final merupakan kombinasi terbobot dari **4 estimator TabPFN v2** independen:
1. **Multi-Seed Stochastic Averaging ($T_3$)**:
   $$\hat{y}_{T_3} = rac{1}{3}\hat{y}_{T42} + rac{1}{3}\hat{y}_{T17} + rac{1}{3}\hat{y}_{T2026}$$
   Melatih 3 instance TabPFN v2 pada 24 fitur mentah (*raw feature space*) dengan variasi seed pengacakan internal ($42, 17, 2026$). Rata-rata ketiga model ini mereduksi variansi inferensi internal *in-context transformer* tanpa mengubah proyeksi geometris data.
2. **Augmentasi Manifold Ortogonal ($TA_{12}$)**:
   Mengekstrak 12 komponen utama via PCA yang di-fit **secara eksklusif hanya pada data latih (zero leakage)**. PCA-12 menangkap **$99.19\%$** variansi kovariansi fitur standar, lalu dikonkatenasikan bersama 24 fitur asli menjadi representasi 36 kolom. Model TabPFN dilatih pada ruang fitur gabungan ini untuk memberikan regularisasi spasial global.
3. **Formulasi Fusi Akhir ($T3\_TA12\_50$)**:
   $$\hat{y}_{	ext{final}} = rac{1}{2}\hat{y}_{T_3} + rac{1}{2}\hat{y}_{TA_{12}} = rac{1}{6}\hat{y}_{T42} + rac{1}{6}\hat{y}_{T17} + rac{1}{6}\hat{y}_{T2026} + rac{1}{2}\hat{y}_{TA_{12}}$$

#### 3. Jaminan Protokol Anti-Leakage & Integritas Submission
* **Zero Target Leakage**: PCA dan transformasi skalar tidak pernah menyentuh target uji maupun vektor uji sebelum fase inferensi.
* **Zero Post-Processing Artifact**: Tanpa clipping batas ekstrem buatan, tanpa kalibrasi skala sepihak, dan tanpa optimasi berbasis feedback public leaderboard yang rentan memicu *private leaderboard shakeup*.
* **Preservasi Invarian Urutan**: ID dan urutan baris dikunci 1:1 terhadap `submission_template.csv` untuk menjamin validitas formal penilaian evaluator kompetisi.

## Bagian 0: Instalasi Lingkungan & Pustaka TabPFN

Arsitektur model ini mengandalkan pustaka resmi `tabpfn` versi modern yang mendukung antarmuka fondasi `ModelVersion.V2`. Seluruh komputasi inferensi dioptimalkan menggunakan backend PyTorch tensor.

In [1]:
!pip install -q tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 17.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 31.3 MB/s eta 0:00:00


## Bagian 1: Inisialisasi Dependensi & Blueprint Global

Pada blok ini dilakukan pemuatan pustaka ilmiah numerik (`numpy`, `pandas`), dekomposisi matriks (`sklearn.decomposition.PCA`), akselerator tensor (`torch`), serta pustaka inti `TabPFNRegressor`. 

Perhatikan deklarasi tata kelola evaluasi:
* Peringatan runtime (`UserWarning`, `ConvergenceWarning`) ditekan untuk menjaga kebersihan output log.
* Tidak ada komponen buatan yang memodifikasi target test secara heuristik.

In [2]:
# ============================================================
# FIKFAIR 2026 — TABPFN T3_TA12_50 FINAL SUBMISSION
# ============================================================
# Ensemble:
#   T42       = raw 24 features, seed 42
#   T17       = raw 24 features, seed 17
#   T2026     = raw 24 features, seed 2026
#   TA12      = raw 24 + PCA(12), seed 42
#
# Weights:
#   T42   : 1/6
#   T17   : 1/6
#   T2026 : 1/6
#   TA12  : 1/2
#
# IMPORTANT:
# - No target leakage
# - No calibration
# - No clipping
# - No leaderboard-based fitting
# - PCA is fitted on TRAIN ONLY
# - Test is used ONLY for final prediction
# ============================================================

import os
import glob
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import torch

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion




## Bagian 2: Konfigurasi Global & Parameter Fusi Ensemble

Blok konfigurasi ini membekukan parameter operasional utama pipeline:
* **`SEEDS = [42, 17, 2026]`**: Seed acak untuk mengeksplorasi variabilitas inisialisasi sampling in-context TabPFN.
* **`N_ESTIMATORS = 8`**: Jumlah estimator internal per TabPFN regressor untuk menyeimbangkan stabilitas estimasi dan kecepatan inferensi.
* **`PCA_COMPONENTS = 12`**: Jumlah koordinat ortogonal yang diekstrak untuk menangkap mayoritas variansi kovariansi fitur.
* **Vektor Bobot ($w$)**: Alokasi bobot simetris $1/6$ untuk ketiga model raw-features, serta bobot $1/2$ ($50\%$) untuk model augmentasi PCA-12.
* **Integritas Checkpoint**: Hash SHA-256 (`2ab5a07d...`) disiapkan sebagai validator integritas bobot neural network sebelum dieksekusi.

In [3]:
# ============================================================
# 1. CONFIG
# ============================================================

SEEDS = [42, 17, 2026]
N_ESTIMATORS = 8
PCA_COMPONENTS = 12

W_T42 = 1/6
W_T17 = 1/6
W_T2026 = 1/6
W_TA12 = 1/2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT = "/kaggle/working/models/tabpfn-v2-regressor.ckpt"
CKPT_URL = (
    "https://huggingface.co/Prior-Labs/TabPFN-v2-reg/"
    "resolve/main/tabpfn-v2-regressor.ckpt"
)

EXPECTED_SHA256 = (
    "2ab5a07d5c41dfe6db9aa7ae106fc6de898326c2765be66505a07e2868c10736"
)

OUTPUT = "/kaggle/working/submission_T3_TA12_50_FINAL.csv"




## Bagian 3: Deteksi Jalur Data Deterministik (*Dynamic Input Resolution*)

Lingkungan kompetisi Kaggle dapat memiliki struktur direktori input yang dinamis bergantung pada penamaan *input dataset version*. Kode ini menggunakan pencarian rekursif deterministik (`glob.glob`) untuk mendeteksi lokasi persis dari `train.csv`, `test.csv`, dan `submission_template.csv` secara otomatis tanpa memerlukan intervensi manual.

In [4]:
# ============================================================
# 2. FIND DATA
# ============================================================

train_candidates = glob.glob(
    "/kaggle/input/**/train.csv",
    recursive=True
)

test_candidates = glob.glob(
    "/kaggle/input/**/test.csv",
    recursive=True
)

template_candidates = glob.glob(
    "/kaggle/input/**/submission_template.csv",
    recursive=True
)

print("Train candidates:")
for p in train_candidates:
    print(" ", p)

print("\nTest candidates:")
for p in test_candidates:
    print(" ", p)

print("\nTemplate candidates:")
for p in template_candidates:
    print(" ", p)


# Pick the first matching files.
# If Kaggle competition folder contains exactly one set, this is deterministic.
TRAIN_PATH = train_candidates[0]
TEST_PATH = test_candidates[0]

if len(template_candidates) == 0:
    raise FileNotFoundError(
        "submission_template.csv tidak ditemukan."
    )

TEMPLATE_PATH = template_candidates[0]

print("\nUsing:")
print("TRAIN   :", TRAIN_PATH)
print("TEST    :", TEST_PATH)
print("TEMPLATE:", TEMPLATE_PATH)




Train candidates:
  /kaggle/input/datasets/arifridho/fikfair-data/train.csv

Test candidates:
  /kaggle/input/datasets/arifridho/fikfair-data/test.csv

Template candidates:
  /kaggle/input/datasets/arifridho/fikfair-data/submission_template.csv

Using:
TRAIN   : /kaggle/input/datasets/arifridho/fikfair-data/train.csv
TEST    : /kaggle/input/datasets/arifridho/fikfair-data/test.csv
TEMPLATE: /kaggle/input/datasets/arifridho/fikfair-data/submission_template.csv



## Bagian 4: Pemuatan Data & Audit Skema / Invarian

Langkah audit integritas data sebelum pemrosesan:
1. **Pemeriksaan Skema**: Memastikan ketersediaan 24 fitur anonim (`feature_01` s.d. `feature_24`) dan target `target` pada dataset latih.
2. **Karakterisasi Target**: Menghitung ringkasan statistik target ACS untuk verifikasi sebaran nilai (mean $51.20$, kuartil interkuartil $[39.69, 60.17]$).
3. **Audit Ketiadaan Nilai Hilang**: Memastikan data bebas dari missing value atau entri non-numerik.

In [5]:
# ============================================================
# 3. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
template = pd.read_csv(TEMPLATE_PATH)

FEATURES = [f"feature_{i:02d}" for i in range(1, 25)]
TARGET = "target"

assert all(c in train.columns for c in FEATURES)
assert TARGET in train.columns
assert all(c in test.columns for c in FEATURES)
assert "record_id" in test.columns
assert "record_id" in template.columns

print("\nShapes:")
print("train   :", train.shape)
print("test    :", test.shape)
print("template:", template.shape)

print("\nTarget:")
print(train[TARGET].describe())




Shapes:
train   : (382, 26)
test    : (96, 25)
template: (96, 2)

Target:
count    382.000000
mean      51.204047
std       12.906033
min       32.586413
25%       39.688575
50%       50.256500
75%       60.166230
max      100.439972
Name: target, dtype: float64



## Bagian 5: Preservasi Urutan Template Submission & Isolasi Matriks Fitur

Ketentuan resmi kompetisi mensyaratkan setiap baris submission cocok secara presisi terhadap urutan `record_id` pada template resmi. 

Pada tahap ini:
* Data uji diindeks berdasarkan `record_id` dan diurutkan ulang (*re-aligned*) agar identik 100% terhadap `submission_template.csv`.
* Kolom penanda identitas `record_id` diisolasi sepenuhnya dan **tidak dimasukkan sebagai fitur model** guna mencegah bias artifisial.
* Matriks fitur latih $X \in \mathbb{R}^{382 	imes 24}$ dan matriks fitur uji $X_{	ext{test}} \in \mathbb{R}^{96 	imes 24}$ dibentuk secara murni.

In [6]:
# ============================================================
# 4. PRESERVE TEMPLATE ORDER
# ============================================================

test_ordered = (
    test
    .set_index("record_id")
    .loc[template["record_id"], FEATURES]
    .copy()
)

assert test_ordered.shape[0] == template.shape[0]

X = train[FEATURES].copy()
y = train[TARGET].copy()

X_test = test_ordered.copy()

print("\nFeature matrix:")
print("X      :", X.shape)
print("X_test :", X_test.shape)




Feature matrix:
X      : (382, 24)
X_test : (96, 24)



## Bagian 6: Manajemen Checkpoint & Audit Kriptografis SHA-256

TabPFN v2 beroperasi menggunakan berkas bobot model pralatih (`tabpfn-v2-regressor.ckpt`). Untuk menjamin reproducibility mutlak dan mencegah kegagalan inferensi akibat unduhan yang terputus (*corrupted binary*), proses ini melakukan:
1. Pengunduhan langsung berkas bobot dari repositori resmi Hugging Face Prior-Labs jika berkas belum ada secara lokal.
2. Verifikasi kriptografis berbasis hashing **SHA-256** secara bertahap (chunk size 1MB).
3. Validasi pencocokan hash terhadap nilai patokan:
   `2ab5a07d5c41dfe6db9aa7ae106fc6de898326c2765be66505a07e2868c10736`.

In [7]:
# ============================================================
# 5. CHECKPOINT
# ============================================================

os.makedirs(os.path.dirname(CKPT), exist_ok=True)

if not os.path.exists(CKPT):
    print("\nDownloading TabPFN checkpoint...")

    import urllib.request

    urllib.request.urlretrieve(
        CKPT_URL,
        CKPT
    )

    print("Download complete.")

else:
    print("\nCheckpoint already exists.")


# SHA-256 verification
sha256 = hashlib.sha256()

with open(CKPT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

print("\nCheckpoint SHA256:")
print(actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "CHECKPOINT SHA256 MISMATCH!\n"
        f"Expected: {EXPECTED_SHA256}\n"
        f"Actual  : {actual_sha256}"
    )

print("Checkpoint verification: PASS")




Download complete.

Checkpoint SHA256:
2ab5a07d5c41dfe6db9aa7ae106fc6de898326c2765be66505a07e2868c10736
Checkpoint verification: PASS



## Bagian 7: Diagnostik Lingkungan Runtime & Akselerator Hardware

Audit sistematis terhadap spesifikasi environment komputasi Kaggle:
* Versi Python, PyTorch, dan Scikit-Learn.
* Status deteksi akselerasi GPU CUDA untuk menjamin eksekusi matriks tensor berjalan optimal.

In [8]:
# ============================================================
# 6. ENVIRONMENT LOG
# ============================================================

print("\n================ ENVIRONMENT ================")
print("Python :", __import__("sys").version)
print("Torch  :", torch.__version__)
print("Device :", DEVICE)

try:
    import sklearn
    print("sklearn:", sklearn.__version__)
except Exception:
    pass

try:
    import tabpfn
    print("TabPFN :", getattr(tabpfn, "__version__", "unknown"))
except Exception:
    pass

print("==============================================")




================ ENVIRONMENT ================
Python : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch  : 2.11.0+cu128
Device : cuda
sklearn: 1.6.1
TabPFN : 9.1.0



## Bagian 8: Pabrikasi Estimator TabPFN v2 (*Model Factory*)

Fungsi `make_tabpfn(seed)` bertindak sebagai factory modular untuk membangkitkan instance `TabPFNRegressor`:
* Menggunakan arsitektur `ModelVersion.V2`.
* Presisi inferensi dikunci pada `torch.float32` guna menjaga stabilitas floating-point numerik.
* Menetapkan `n_estimators = 8` dan pengacakan acak terarah berbasis parameter `seed`.

In [9]:
# ============================================================
# 7. TABPFN FACTORY
# ============================================================

def make_tabpfn(seed):
    print(
        f"Creating TabPFN | seed={seed} | "
        f"estimators={N_ESTIMATORS}"
    )

    return TabPFNRegressor.create_default_for_version(
        ModelVersion.V2,
        model_path=CKPT,
        device=DEVICE,
        inference_precision=torch.float32,
        n_estimators=N_ESTIMATORS,
        random_state=seed,
        n_preprocessing_jobs=1,
    )




## Bagian 9: Komponen 1/4 — Estimator Raw Features Seed 42 (`T42`)

Estimator pertama dilatih pada 24 fitur mentah asli menggunakan `seed=42`. Model ini menangkap pola non-parametrik dasar antara variabel iklim/lingkungan dan target biaya hidup ACS.

* Fitur: 24 fitur mentah
* Random State: 42
* Bobot dalam Final Ensemble: $1/6 pprox 16.67\%$

In [10]:
# ============================================================
# 8. T42
# ============================================================

print("\n================================================")
print("MODEL 1/4 — T42")
print("raw 24 features | seed=42")
print("================================================")

model_T42 = make_tabpfn(42)

model_T42.fit(
    X,
    y
)

pred_T42 = model_T42.predict(X_test)

pred_T42 = np.asarray(pred_T42, dtype=float)

print(
    "T42 prediction:",
    pred_T42.shape,
    "mean=",
    pred_T42.mean(),
    "std=",
    pred_T42.std()
)




MODEL 1/4 — T42
raw 24 features | seed=42
Creating TabPFN | seed=42 | estimators=8
T42 prediction: (96,) mean= 50.430793862079746 std= 11.023969910583414



## Bagian 10: Komponen 2/4 — Estimator Raw Features Seed 17 (`T17`)

Estimator kedua dilatih dengan konfigurasi identik pada 24 fitur mentah, namun menggunakan inisialisasi pengacakan sampling `seed=17`. Pengacakan ini memberikan perspektif inferensi alternatif pada titik-titik data batas (*boundary cases*).

* Fitur: 24 fitur mentah
* Random State: 17
* Bobot dalam Final Ensemble: $1/6 pprox 16.67\%$

In [11]:
# ============================================================
# 9. T17
# ============================================================

print("\n================================================")
print("MODEL 2/4 — T17")
print("raw 24 features | seed=17")
print("================================================")

model_T17 = make_tabpfn(17)

model_T17.fit(
    X,
    y
)

pred_T17 = model_T17.predict(X_test)

pred_T17 = np.asarray(pred_T17, dtype=float)

print(
    "T17 prediction:",
    pred_T17.shape,
    "mean=",
    pred_T17.mean(),
    "std=",
    pred_T17.std()
)




MODEL 2/4 — T17
raw 24 features | seed=17
Creating TabPFN | seed=17 | estimators=8
T17 prediction: (96,) mean= 50.407399725160154 std= 11.056228033734511



## Bagian 11: Komponen 3/4 — Estimator Raw Features Seed 2026 (`T2026`)

Estimator ketiga melengkapi trio multi-seed raw features ($T_3$) dengan `seed=2026`. Rata-rata dari ketiga seed ini secara empiris terbukti mereduksi variansi residual in-context tanpa mengorbankan ketepatan titik rata-rata.

* Fitur: 24 fitur mentah
* Random State: 2026
* Bobot dalam Final Ensemble: $1/6 pprox 16.67\%$

In [12]:
# ============================================================
# 10. T2026
# ============================================================

print("\n================================================")
print("MODEL 3/4 — T2026")
print("raw 24 features | seed=2026")
print("================================================")

model_T2026 = make_tabpfn(2026)

model_T2026.fit(
    X,
    y
)

pred_T2026 = model_T2026.predict(X_test)

pred_T2026 = np.asarray(pred_T2026, dtype=float)

print(
    "T2026 prediction:",
    pred_T2026.shape,
    "mean=",
    pred_T2026.mean(),
    "std=",
    pred_T2026.std()
)




MODEL 3/4 — T2026
raw 24 features | seed=2026
Creating TabPFN | seed=2026 | estimators=8
T2026 prediction: (96,) mean= 50.461974570735066 std= 10.976671376668412



## Bagian 12: Rekayasa Manifold Unsupervised — Dekomposisi PCA-12

### Landasan Geometris & Pencegahan Kebocoran Target
Eksplorasi data awal membuktikan adanya multikolinearitas kuat antar-fitur (terdapat 11 pasangan fitur dengan $|r| \ge 0.85$). 
1. **Fit Eksklusif pada Data Latih**: `PCA(n_components=12)` di-fit **hanya** pada matriks latih $X$. Proyeksi data uji $X_{	ext{test}}$ dilakukan secara independen menggunakan parameter transformasi yang telah dibekukan.
2. **Konsentrasi Variansi**: 12 komponen utama merangkum **$99.1934\%$** total variansi informasi fitur.
3. **Ruang Fitur Gabungan ($36	ext{ Fitur}$)**: Menggabungkan 24 fitur asli dengan 12 fitur terproyeksi ortogonal. Hal ini mempertahankan resolusi informasi tingkat atomik sekaligus menyajikan sumbu variansi global bagi model.

In [13]:
# ============================================================
# 11. PCA12
# ============================================================
# PCA is FIT ONLY on training data.
#
# No target is used.
# Test is transformed using the already-fitted PCA.
# ============================================================

print("\n================================================")
print("PCA12 PREPROCESSING")
print("================================================")

pca12 = PCA(
    n_components=PCA_COMPONENTS,
    random_state=42
)

X_pca12 = pca12.fit_transform(X)
X_test_pca12 = pca12.transform(X_test)

print("X_pca12      :", X_pca12.shape)
print("X_test_pca12 :", X_test_pca12.shape)

explained = pca12.explained_variance_ratio_

print(
    "PCA12 explained variance:",
    explained.sum()
)


# Combine raw + PCA12
X_TA12 = np.concatenate(
    [
        X.to_numpy(dtype=float),
        X_pca12
    ],
    axis=1
)

X_test_TA12 = np.concatenate(
    [
        X_test.to_numpy(dtype=float),
        X_test_pca12
    ],
    axis=1
)

print("TA12 train :", X_TA12.shape)
print("TA12 test  :", X_test_TA12.shape)




PCA12 PREPROCESSING
X_pca12      : (382, 12)
X_test_pca12 : (96, 12)
PCA12 explained variance: 0.9919342355740959
TA12 train : (382, 36)
TA12 test  : (96, 36)



## Bagian 13: Komponen 4/4 — Estimator Dual-Representation (`TA12`)

Model `TA12` dilatih pada matriks representasi ganda 36 kolom ($24	ext{ Raw} + 12	ext{ PCA}$) dengan `seed=42`. 

Pada audit validasi lokal, penambahan cabang dekomposisi ortogonal ini memberikan sinyal regularisasi yang sangat kuat, memangkas error prediksi pada kelompok data ekstrim (*far-distance records*) tanpa mengacaukan prediksi di kelompok tengah.

* Fitur: 36 fitur gabungan
* Random State: 42
* Bobot dalam Final Ensemble: $1/2 = 50.00\%$

In [14]:
# ============================================================
# 12. TA12
# ============================================================

print("\n================================================")
print("MODEL 4/4 — TA12")
print("raw24 + PCA12 | seed=42")
print("================================================")

model_TA12 = make_tabpfn(42)

model_TA12.fit(
    X_TA12,
    y
)

pred_TA12 = model_TA12.predict(
    X_test_TA12
)

pred_TA12 = np.asarray(pred_TA12, dtype=float)

print(
    "TA12 prediction:",
    pred_TA12.shape,
    "mean=",
    pred_TA12.mean(),
    "std=",
    pred_TA12.std()
)




MODEL 4/4 — TA12
raw24 + PCA12 | seed=42
Creating TabPFN | seed=42 | estimators=8
TA12 prediction: (96,) mean= 50.346655756168 std= 11.02443794570472



## Bagian 14: Fusi Ensemble Terbobot Variansi (`T3_TA12_50`)

### Formulasi Fusi Akhir
Prediksi final merupakan kombinasi linear terbobot:
$$\hat{y}_{	ext{final}} = rac{1}{6} \hat{y}_{T42} + rac{1}{6} \hat{y}_{T17} + rac{1}{6} \hat{y}_{T2026} + rac{1}{2} \hat{y}_{TA12}$$

### Analisis Karakteristik Distribusi Prediksi
* **Mean ($\mu$)**: $pprox 50.39$ (sangat selaras terhadap mean target aktual latih $51.20$).
* **Standar Deviasi ($\sigma$)**: $pprox 11.01$ (konservatif terhadap deviasi aktual $12.91$).
* **Rentang Nilai ($[\min, \max]$)**: $[34.27, 90.78]$ berada sepenuhnya dalam domain realistis target latih $[32.59, 100.44]$. Tidak ditemukan anomali nilai negatif atau ledakan ekstrapolasi.

In [15]:
# ============================================================
# 13. FINAL ENSEMBLE
# ============================================================

print("\n================================================")
print("FINAL ENSEMBLE — T3_TA12_50")
print("================================================")

prediction = (
    W_T42   * pred_T42
    + W_T17 * pred_T17
    + W_T2026 * pred_T2026
    + W_TA12 * pred_TA12
)

prediction = np.asarray(prediction, dtype=float)

assert len(prediction) == len(template)
assert np.isfinite(prediction).all()

print("\nWeights:")
print("T42   :", W_T42)
print("T17   :", W_T17)
print("T2026 :", W_T2026)
print("TA12  :", W_TA12)

print("\nFinal prediction:")
print("mean :", prediction.mean())
print("std  :", prediction.std())
print("min  :", prediction.min())
print("max  :", prediction.max())




FINAL ENSEMBLE — T3_TA12_50

Weights:
T42   : 0.16666666666666666
T17   : 0.16666666666666666
T2026 : 0.16666666666666666
TA12  : 0.5

Final prediction:
mean : 50.39002257107982
std  : 11.005338688794264
min  : 34.26604718450612
max  : 90.77511756725858



## Bagian 15: Pembentukan Berkas Submission & Verifikasi Format

Hasil prediksi digabungkan dengan `record_id` dari template submission resmi dan diekspor ke file CSV:
`/kaggle/working/submission_T3_TA12_50_FINAL.csv`.

Dilakukan peninjauan langsung terhadap 10 baris pertama untuk memastikan format data, penamaan kolom, dan konsistensi numerik.

In [16]:
# ============================================================
# 14. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "record_id": template["record_id"],
    "prediction": prediction
})

assert list(submission.columns) == [
    "record_id",
    "prediction"
]

assert submission["record_id"].equals(
    template["record_id"]
)

assert submission["record_id"].is_unique
assert submission["prediction"].notna().all()

submission.to_csv(
    OUTPUT,
    index=False
)

print("\n================================================")
print("SUBMISSION CREATED")
print("================================================")

print(OUTPUT)

print("\nFirst 10 rows:")
display(submission.head(10))

print("\nSubmission shape:")
print(submission.shape)

print("\nPrediction summary:")
print(submission["prediction"].describe())




SUBMISSION CREATED
/kaggle/working/submission_T3_TA12_50_FINAL.csv

First 10 rows:


,record_id,prediction
0,D0263,90.775118
1,D0280,73.162698
2,D0206,44.483977
3,D0133,37.723139
4,D0352,61.201211
5,D0339,78.371591
6,D0261,68.098507
7,D0409,70.622529
8,D0411,65.698592
9,D0144,59.467743



Submission shape:
(96, 2)

Prediction summary:
count    96.000000
mean     50.390023
std      11.063110
min      34.266047
25%      40.978158
50%      49.484244
75%      59.361647
max      90.775118
Name: prediction, dtype: float64



## Bagian 16: Audit Sanity Check Akhir & Jaminan Validitas Submission

Audit pertahanan terakhir sebelum berkas diserahkan ke sistem evaluasi lomba:
* **Dimensi**: Memastikan tepat 96 baris data uji.
* **Keunikan Identitas**: Memastikan 96 `record_id` unik tanpa data duplikat atau hilang.
* **Integritas Numerik**: Memastikan tidak terdapat `NaN`, `None`, atau nilai infinite ($\pm\infty$).
* **Kesesuaian Skema**: Memastikan urutan kolom tepat `['record_id', 'prediction']`.

Status verifikasi: **PASS**. Berkas siap diunggah ke portal kompetisi Kaggle.

In [17]:
# ============================================================
# 15. FINAL SANITY CHECK
# ============================================================

print("\n================================================")
print("FINAL SANITY CHECK")
print("================================================")

print("Rows              :", len(submission))
print("Unique record_id  :", submission["record_id"].nunique())
print("NaN predictions   :", submission["prediction"].isna().sum())
print("Infinite values   :", np.isinf(submission["prediction"]).sum())
print(
    "Column order      :",
    list(submission.columns)
)

print("\nDONE.")
print("Upload this file to Kaggle:")
print(OUTPUT)

FINAL SANITY CHECK
Rows              : 96
Unique record_id  : 96
NaN predictions   : 0
Infinite values   : 0
Column order      : ['record_id', 'prediction']

DONE.
Upload this file to Kaggle:
/kaggle/working/submission_T3_TA12_50_FINAL.csv


---

## Catatan Teknis & Analisis Risiko Pasca-Submission

### Mengapa Pendekatan Ini Tangguh Menghadapi Private Leaderboard Shakeup?
1. **Tidak Bergantung pada Tuning Leaderboard Publik**: Publik Leaderboard hanya merepresentasikan **29 sampel** (~$30\%$ data uji), sedangkan penentu pemenang akhir adalah Private Leaderboard dengan **67 sampel**. Model yang overfit pada leaderboard publik rentan ambruk pada saat unblinding private. Arsitektur `T3_TA12_50` dirancang berbasis audit validasi lokal 30-fold dan konfirmasi bebas bocor.
2. **Ketiadaan Post-Processing Buatan**: Banyak peserta melakukan pemotongan batas (*threshold clipping*) atau penyesuaian skala (*mean shifting*) untuk memburu desimal di public leaderboard. Seluruh eksperimen internal tim membuktikan bahwa intervensi heuristik tersebut meningkatkan error pada ekor target tinggi. Model ini mempertahankan estimasi murni berbasis kepadatan posterior TabPFN.
3. **Sinergi Dual-Representation**: Menggabungkan dekomposisi global ortogonal (PCA-12) dan koordinat spasial lokal (Raw 24) menghadirkan estimasi yang stabil terhadap pergeseran kovariat (*covariate shift*).